# Three-chamber social walkthrough: from a session folder to results

You recorded calcium imaging with a miniscope while a mouse explored a three-chamber arena with a
stimulus animal under the left cup. You have a folder with the activity, two timestamp files (miniscope and
behaviour camera), tracked position, head orientation, and hand-scored zone and investigation epochs.
This notebook takes that folder to a table of which cells care about investigating, which are place cells,
and the figures to show for it. It uses the simulated three-chamber fixture shipped with CaBaNA
(`tests/data/simulated_3chamber`, 75 s, 180 cells), so every number is reproducible.

The chain: **load** the folder, **check** the clocks, **derive** behaviour from geometry and compare it with
your annotations, **QC** the cells, run **one pipeline** of analyses against a **null**, then read the
**table** and **classify** cells.

Running in Colab? Run the next cell first; locally, skip it.

In [ ]:
# Colab / fresh environment only: install CaBaNA and fetch the example session folder.
import pathlib, subprocess, sys

if not pathlib.Path("../tests/data/simulated_3chamber").exists() and not pathlib.Path("simulated_3chamber").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/miniscope/CaBaNA@framework-demo"], check=True)
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "framework-demo", "--filter=blob:none",
                    "--sparse", "https://github.com/miniscope/CaBaNA", "cabana_repo"], check=True)
    subprocess.run(["git", "-C", "cabana_repo", "sparse-checkout", "set",
                    "tests/data/simulated_3chamber"], check=True)
    subprocess.run(["cp", "-r", "cabana_repo/tests/data/simulated_3chamber", "simulated_3chamber"], check=True)

In [ ]:
import pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import cabana
from cabana import AnalysisSpec, Pipeline, ShuffleConfig, build
from cabana.loaders import load_session_dir

pd.set_option("display.width", 160)
# the session folder: in the repo it is ../tests/data/simulated_3chamber, in Colab ./simulated_3chamber
SESSION_DIR = next(p for p in (pathlib.Path("../tests/data/simulated_3chamber"),
                               pathlib.Path("simulated_3chamber")) if p.exists())
N_SHUFFLES = 100   # 200 or more for a real analysis
print(sorted(p.name for p in SESSION_DIR.iterdir()))

## 1. Load the session folder

`session.yaml` sits next to the data and says what each file is: which `.npy` is the activity and whether it is
cells x frames, which CSV holds the miniscope timestamps and which the camera's, where `-1` means a lost
position, the arena zones and the cups (in cm), and which epoch files are annotations.
`load_session_dir` reads it and returns a **`Session`**: activity on the miniscope time axis, behaviour
resampled onto that axis, and annotations as state spaces. Nothing is silently fixed: the two clocks are
reported so you can see whether frames were dropped.

In [ ]:
ses = load_session_dir(SESSION_DIR)

for clock, rep in ses.meta["timing"].items():
    print(f"{clock:8s} frames={rep['n_frames']}  fs={rep['fs_mean']:.2f} Hz  "
          f"dropped={rep['n_dropped_frames']}  jitter={rep['jitter_sd_s'] * 1000:.2f} ms")
print(ses.info)
print(ses.meta["arena"])
print(f"{ses.n_cells} cells x {ses.n_time} frames, activity kind: {ses.activity_kind}")

A **`StateSpace`** is the framework's word for "what the animal was doing at each frame": a set of named states
(`left / center / right`, or `investigate_stimulus / investigate_empty`) plus a **support**, the frames where
the label is actually known. Frames with a lost position are outside the support and are never silently
counted as "not investigating". The annotated zones and investigation epochs came in as state spaces:

In [ ]:
for name, space in ses.states.items():
    print(f"{name:14s} states={space.state_names}  exclusive={space.exclusive}  "
          f"frames in each state={space.onehot.sum(axis=0).tolist()}")
cabana.cell_qc_table(ses).describe().round(2)

In [ ]:
ses = build("transform", {"name": "speed"}).apply(ses)          # cm/s from x, y
cabana.plots.plot_behavior_overview(ses);

In [ ]:
# the arena in cm: zones, cups, and where the head points (every 25th frame)
from matplotlib.patches import Circle, Rectangle
arena = ses.meta["arena"]
fig, ax = plt.subplots(figsize=(9, 5))
for name, (x0, y0, x1, y1) in arena["zones"].items():
    ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, alpha=0.15, color={"left": "C0", "center": "0.5", "right": "C1"}[name]))
for name, e in arena["enclosures"].items():
    ax.add_patch(Circle(e["center"], e["radius"], fill=False, ls="--"))
    ax.text(*e["center"], f"{name}\n({e['contents']})", ha="center", fontsize=8)
x, y, hd = ses.signals["x"], ses.signals["y"], ses.signals["head_direction"]
ax.plot(x, y, lw=0.6, color="0.3")
k = np.arange(0, ses.n_time, 25)
ax.quiver(x[k], y[k], np.cos(hd[k]), np.sin(hd[k]), color="crimson", scale=30, width=0.003)
ax.set_xlim(-10, 70); ax.set_ylim(-1, 31); ax.set_aspect("equal"); ax.set_xlabel("x (cm)"); ax.set_ylabel("y (cm)");

## 2. Derive behaviour from geometry, and check your annotations

Why derive it at all, when you already scored it by hand? Because the hand-scored epochs encode a rule
("within about 4 cm of the cup, head pointing at it") that you applied by eye. A geometry rule applied to
the tracked position is exact, repeatable, and can be re-run with a different distance or angle. Comparing the
two is the "do my annotations match my own definition" check, and it is also how you would score a second
animal without watching the video.

`zone_occupancy` assigns each frame to the zone rectangle in `session.yaml`; `investigation_from_geometry`
marks a frame as investigating a cup when the animal is within `max_distance` of the cup's edge *and* its head is
within `max_angle_deg` of facing it (defaults 4 cm and 45 degrees).

In [ ]:
ses = build("transform", {"name": "zone_occupancy", "signal": "", "zones_from_meta": True,
                          "output": "zones_geo"}).apply(ses)
ses = build("transform", {"name": "investigation_from_geometry",
                          "output": "investigation_geo"}).apply(ses)

# frame-by-frame agreement of the derived and annotated investigation states, and bout counts
PAIR = {"left_cup": "investigate_stimulus", "right_cup": "investigate_empty"}
geo, ann = ses.states["investigation_geo"], ses.states["investigation"]
gb = cabana.states_to_bouts(geo, ses.fs, t0=ses.t[0])
ab = cabana.states_to_bouts(ann, ses.fs, t0=ses.t[0])
ok = geo.support & ann.support
rows = []
for cup, label in PAIR.items():
    g = geo.onehot[:, geo.state_names.index(cup)]
    a = ann.onehot[:, ann.state_names.index(label)]
    gi, ai = [i for i, n in enumerate(gb.labels) if n == cup], [i for i, n in enumerate(ab.labels) if n == label]
    # an annotated bout is "found" if a geometry bout starts and stops within 0.5 s of it
    found = sum(any(abs(gb.start_s[j] - ab.start_s[i]) <= 0.5 and abs(gb.stop_s[j] - ab.stop_s[i]) <= 0.5
                    for j in gi) for i in ai)
    rows.append(dict(cup=cup, frame_agreement=(g[ok] == a[ok]).mean(), bouts_geometry=len(gi),
                     bouts_annotated=len(ai), annotated_bouts_found=found))
pd.DataFrame(rows).round(4)

Frame agreement near 1 and every annotated bout found means your hand scoring and the geometry rule describe the
same behaviour; this is a simulated session, so agreement is as good as it gets. On real data, look at where
they disagree before choosing which one to analyse: the geometry version is the one you can rerun.

## 3. Cell QC

Neural input here is already deconvolved (a non-negative activity estimate), so no event detection is needed.
`cell_qc_table` showed the spread of firing rates above. Dropping the quietest cells and any with missing
frames is a preprocessing step, `select_cells`, which records the dropped ids; the thresholds are yours to
choose, and they are chosen here before any statistics are computed.

In [ ]:
qc = cabana.cell_qc_table(ses)
min_rate = float(np.round(qc["mean_rate_hz"].quantile(0.05), 2))
qc_step = {"name": "select_cells", "min_rate_hz": min_rate, "min_fraction_finite": 0.9, "keep_at_least": 10}
qc_ses = build("preprocess", qc_step).apply(ses)
sel = qc_ses.meta["cell_selection"]
print(f"min_rate_hz >= {min_rate}: {sel['n_before']} -> {sel['n_after']} cells; dropped {sel['dropped_ids']}")

## 4. One pipeline: what the analyses ask, and what the null is

A **`Pipeline`** bundles the steps and analyses so the whole thing is one object you can save next to the results:

- **transforms** build state spaces from signals: `bin_grid` turns x, y into a grid of 5 cm bins (a *position*
  state space);
- **preprocess** cleans the session: check the clocks, keep the frames covered by both position and head
  data, drop the QC-failed cells, mask out frames where the animal moves slower than 2 cm/s (resting
  activity is a different question from place coding);
- **selectors** choose a parameter from the data without looking at p-values; `auto_null_timescale` sets how far
  the null may shift activity from the autocorrelation;
- **analyses** are a metric on a state space plus a null.

**What is the null?** Every metric is computed on the real data and again on many surrogate datasets in which the
relationship between activity and behaviour has been broken while keeping each trace's own structure. If the real
value sits far outside the surrogate values, that cell is called significant. Two nulls are used here:
`bout_shuffle` moves whole investigation bouts around (the right question when behaviour comes in bouts, and with
only a few bouts), and the default `circular_shift` slides the activity in time by a random amount of at least
`min_shift_s` (right for position, where behaviour is continuous). `correction="max_stat"` then corrects for a cell
having several states to be "significant" in.

In [ ]:
bout = ShuffleConfig(n_shuffles=N_SHUFFLES, null={"name": "bout_shuffle"}, correction="max_stat", seed=0)
circ_c = ShuffleConfig(n_shuffles=N_SHUFFLES, min_shift_s=20.0, correction="max_stat", seed=0)
circ = ShuffleConfig(n_shuffles=N_SHUFFLES, min_shift_s=20.0, seed=0)
w, h = arena["size"]

pipe = Pipeline(
    selectors=[{"name": "auto_null_timescale"}],
    transforms=[{"name": "bin_grid", "signals": ["x", "y"], "bin_size": 5.0, "output": "position",
                 "extent": [(0.0, w), (0.0, h)]}],
    preprocess=[{"name": "timing_check"},
                {"name": "trim_to_overlap", "signals": ["position_covered", "head_covered"]},
                qc_step,
                {"name": "speed_threshold", "min_speed": 2.0}],
    analyses=[
        # does a cell fire more during investigation than outside it? (0.5 = no preference)
        AnalysisSpec(metric={"name": "auroc"}, states="investigation", shuffle=bout),
        AnalysisSpec(metric={"name": "mean_per_state", "statistic": "difference"},
                     states="investigation", shuffle=bout),
        # average activity around the onset of each investigation bout
        AnalysisSpec(metric={"name": "peth", "pre_s": 4.0, "post_s": 4.0, "baseline_s": (-4.0, -1.0),
                             "response_s": (0.0, 1.5), "min_duration_s": 1.0, "min_gap_s": 1.0},
                     states="investigation", shuffle=circ_c),
        # place coding: spatial information and split-half stability, moving frames only
        AnalysisSpec(metric={"name": "information_content"}, states="position", shuffle=circ, masks=["speed"]),
        AnalysisSpec(metric={"name": "split_half_stability"}, states="position", shuffle=circ, masks=["speed"]),
        # how much does the zone the animal is in tell you about the cell's activity?
        AnalysisSpec(metric={"name": "mutual_information"}, states="zones_geo", shuffle=circ_c),
    ],
)
session, rs = pipe.run(ses, progress=False)
print(session.n_cells, "cells,", session.n_time, "frames after preprocessing")
for s in rs.selections:
    print("selected:", s["path"], "=", s["value"], f"[{s['method']}]")

## 5. Reading the table

`rs.to_dataframe()` is one row per cell and state: the observed `value`, the permutation `p_value` and
whether it is `significant` at alpha 0.05; with a correction, `p_adj` and `significant_adj` correct across a
cell's states, and **those are the ones to report**. A cell is "significant" for an analysis if it is in any
state.

In [ ]:
df = rs.to_dataframe()
summary = pd.DataFrame([{
    "analysis": name,
    "cells": d["cell"].nunique(),
    "significant (raw)": int(d.groupby("cell")["significant"].any().sum()),
    "significant (adjusted)": int(d.groupby("cell")["significant_adj"].any().sum()),
} for name, d in df.groupby("analysis", sort=False)])
print(summary.to_string(index=False))
df[df["analysis"] == "auroc[investigation]"].sort_values("p_value").head(8).round(3)

The AUROC is the probability that a random investigation frame has higher activity than a random non-investigation
frame, so 0.5 means no preference, above 0.5 means excited, below means suppressed. The heatmap shows all cells
sorted by their preferred state; white dots mark the significant ones. Note that a *state with a single bout*
(`investigate_empty` here lasts one bout) can reach significance only through that one episode: treat it as
suggestive and score more sessions before relying on it.

In [ ]:
au = rs["auroc[investigation]"]
cabana.plots.plot_tuning_heatmap(au, sort_by="peak");

The null for the three strongest cells: the red line is the real value, the grey bars are the surrogates and the dashed lines bound 95% of them.

In [ ]:
strength = np.nanmax(np.abs(np.nan_to_num(au.values - 0.5)), axis=1)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), constrained_layout=True)
for ax, c in zip(axes, np.argsort(-strength)[:3]):
    k = int(np.nanargmax(np.abs(np.nan_to_num(au.values[c] - 0.5))))
    cabana.plots.plot_shuffle_distribution(au, int(c), state=k, ax=ax)

Average activity around the onset of investigation (dashed line); orange marks the response window. `investigate_empty` has too few bouts for a PETH.

In [ ]:
cabana.plots.plot_peth(rs["peth[investigation]"], session, ncols=4);

**Place coding.** Spatial information and split-half stability (does the map from the first half of the session
match the second?) both have to be significant for the usual place-cell label. Rate maps below are the six
cells with the highest information, significant ones first. With only 75 s of data the maps are coarse; a real
session of many minutes gives much cleaner fields.

In [ ]:
ic = rs["information_content[position]"]
order = np.lexsort((-np.nan_to_num(ic.values, nan=-np.inf), ~ic.shuffle.significant))
cabana.plots.plot_rate_maps(session, ic, [int(i) for i in order[:6]], ncols=3);

## 6. Classify cells and save

Criteria combine tests on the results into named cell classes: a **place cell** has significant information and
stability; a cell is **excited** by a state when its AUROC is above the 97.5th percentile of its null, and
**inhibited** below the 2.5th. Classes may overlap.

In [ ]:
rules = {"place_cell": cabana.place_cell_criteria("position")}
rules.update(cabana.responsive_cell_criteria(cabana.ResultSet(results=[au]), "investigation"))
cls = cabana.classify(rs, rules)
print(cls.counts())
cabana.plots.plot_classification(cls);

In [ ]:
import json, tempfile
out = pathlib.Path(tempfile.mkdtemp(prefix="three_chamber_"))
rs.save(out / "results.zip")                 # reload with cabana.ResultSet.load
session.save(out / "session.zip")            # reload with cabana.Session.load
df.to_csv(out / "results.csv", index=False)
(out / "selections.json").write_text(json.dumps(rs.selections, indent=2, default=str))
print(out, sorted(p.name for p in out.iterdir()))

## To run on your own data

1. Write a `session.yaml` from the template, `cabana.loaders.write_session_dir_template("my_session")`, and
   point its file names at your activity, timestamps, position, head orientation and annotation files.
2. `load_session_dir("my_session")`, then check the timing report and the QC table before anything else.
3. Adjust `arena.zones` and `arena.enclosures` (cm, in the same frame as your position), and the state names
   in the cells above if your annotation labels differ. The same chain is available as a script,
   `python examples/three_chamber.py --session my_session --out my_out`.

More: `docs/LOADERS.md` ("A session folder with a config") and `docs/BRING_YOUR_OWN_DATA.md`.